# The NVIDIA Collectives Communications Library (NCCL)

## Introduction

NCCL (pronounced “Nickel”) is a library of multi-GPU collective and point-to-point communication primitives that are topology-aware and are designed to be light-weight, depending only on the standard C++ and CUDA libraries. NCCL can be deployed in single-process or multi-process applications, handling required inter-process communication transparently. Moreover, the API is quite similar to MPI and provides the functionality of most-used MPI primitives.

In general, NCCL is optimized for high bandwidth and low latency over PCIe and NVLink high speed interconnect for intra-node communication and sockets and InfiniBand for inter-node communication. NCCL—allows CUDA applications and DL frameworks in particular—to efficiently use multiple GPUs without having to implement complex communication algorithms and adapt them to every platform.

**Relevance to Deep Learning:** The NVIDIA AI libraries in CUDA-X depend on NCCL to provide a programming abstraction that is highly tuned for each platform and topology-aware through advanced topology detection, generic path search, and algorithms optimized for NVIDIA architectures. Consequently, developers using deep learning frameworks can rely on NCCL’s highly optimized, MPI compatible and topology aware routines, to take full advantage of all available GPUs within and across multiple nodes.

### NCCL compared with CUDA-aware MPI

Here are some differenciating factors of NCCL compared to CUDA-aware MPI:

* NCCL APIs are initiated from the CPU, but they execute on the GPU and they move or exchange data among GPU memories whereas MPI executes entirely on CPU. 
* It also uses CUDA Stream semantics with a stream parameter while MPI (CUDA-aware or otherwise) is not stream-aware.
* NCCL requires a parent communication framework like MPI or SHMEM. 
* Unlike MPI, it does not have tags.
* NCCL is most optimized for collective communication and is more efficient than MPI in dense systems like DGX.

### Architecture Overview 

Here's an overview of NCCL's architecture:

![nccl_architecture](../../images/nccl_architecture.png)

NCCL maintains separate threads in GPU and CPU for intra-node and network-bound communication, respectively. Crucially, it detects complex multi-node and intra-node topologies at runtime and generates optimal paths for data transfer between two GPUs dynamically.

For example, recall that within a DGX-1V node, device-to-device `cudaMemcpy` between GPU 0 and GPU 7 requires data movement through PCIe bus and SMP interconnect, both of which are bandwidth constrained compared to NVLink. This route is denoted by the blue path below.

![nccl_dgx1_topology](../../images/nccl_dgx1_topology.png)

Note that there isn't a direct NVLink connection between GPUs 0 and 7 on DGX with 8 V100 GPUs. NCCL, however, can utilize GPU 4 to establish a one-hop NVLink-based connection between GPU 0 and 7 as deonted by the red path above. NCCL employs many such optimizations transparently to the users that ultimately results in higher network utilization and application performance.

## Using NCCL

### Communicator Initialization

The NCCL API closely follows MPI. Before performing data transfer operations, a communicator object must be created for each GPU. The communicators identify the set of GPUs that will communicate and maps the communication paths between them. We call the set of associated communicators a "clique". The most general method to initialize communicators in NCCL is to call `ncclCommInitRank()` once for each GPU:

```c
ncclComm_t nccl_comm;
NCCL_CALL(ncclCommInitRank(&nccl_comm, nGPUs, nccl_uid, rank));
```

This function assumes that the GPU belonging to the specified rank has already been selected using `cudaSetDevice()`. `nGPUs` is the number of GPUs in the clique. `nccl_uid` allows the ranks of the clique to find each other. The same `nccl_uid` must be used by all ranks. To achieve this, call `ncclGetUniqueId()` in one rank and broadcast the resulting `nccl_uid` to the other ranks of the clique using MPI as follows:

```c
ncclUniqueId nccl_uid;
if (rank == 0) {
    ncclGetUniqueId(&nccl_uid);
}
MPI_Bcast(&nccl_uid, sizeof(ncclUniqueId), MPI_BYTE, 0, MPI_COMM_WORLD);
```

The last argument to `ncclCommInitRank()`, `rank`, specifies the index of the current GPU within the clique. It must be unique for each rank in the clique and in the range $[0, nGPUs)$.

Internally, `ncclInitRank()` performs a synchronization between all communicators in the clique. As a consequence, it must be called from a different host thread for each GPU, or from separate processes (e.g., MPI ranks). 

Thus, in a multi-node program that uses MPI, the steps to initialize NCCL communicator are as follows:

1. Get `rank` and `size` from `MPI_Comm_rank` and `MPI_Comm_size` functions, respectively. Now, `nGPUs`, which is the total number of GPUs used, will be equal to `size`, as we are using the single-GPU per rank communication model.
2. Get the unique clique ID, `nccl_uid`, using `ncclGetUniqueId` function on rank 0 and broadcast it using `MPI_Bcast`.
3. Get the local rank, `local_rank`, of the process within a node using a local MPI communicator and `MPI_Comm_split_type`, `MPI_Comm_rank`, and `MPI_Comm_free` functions as done in previous labs.
4. Use `cudaSetDevice` to set the current GPU as `local_rank`.
5. Use `ncclCommInitRank` function to initialize NCCL communicator on all ranks.

The code for this process is as follows:

```c
MPI_Init(&argc, &argv);
int rank;
MPI_Comm_rank(MPI_COMM_WORLD, &rank);
int size;
MPI_Comm_size(MPI_COMM_WORLD, &size);

ncclUniqueId nccl_uid;
if (rank == 0) ncclGetUniqueId(&nccl_uid);
MPI_Bcast(&nccl_uid, sizeof(ncclUniqueId), MPI_BYTE, 0, MPI_COMM_WORLD);

int local_rank = -1;
{
    MPI_Comm local_comm;
    MPI_Comm_split_type(MPI_COMM_WORLD, MPI_COMM_TYPE_SHARED, rank, MPI_INFO_NULL, &local_comm);
    MPI_Comm_rank(local_comm, &local_rank);
    MPI_Comm_free(&local_comm);
}
cudaSetDevice(local_rank);

ncclComm_t nccl_comm;
ncclCommInitRank(&nccl_comm, size, nccl_uid, rank);
```

### Group Calls

Several data transfer calls can be merged together using NCCL Groups by encapsulating memory copy operations between `ncclGroupStart()` and `ncclGroupEnd()` function calls. This is needed for three purposes: managing multiple GPUs from one thread (to avoid deadlocks), aggregating communication operations to improve performance, or merging multiple send/receive point-to-point operations.

It is advisible to always encapsulate NCCL communication functions within group calls.

### API Summary

We now give an API overview and list below the essential functions like communicator creation/ destruction, commonly used point-to-point communication functions, a couple of collective communication functions, and aggregating communication functions using group calls for performance optimization. 

```bash
// Communicator creation
ncclGetUniqueId(ncclUniqueId* commId);
ncclCommInitRank(ncclComm_t* comm, int nranks, ncclUniqueId commId, int rank);

// Communicator destruction
ncclCommDestroy(ncclComm_t comm);

// Point-to-point communication
ncclSend(void* sbuff, size_t count, ncclDataType_t type, int peer, ncclComm_t comm, cudaStream_t stream);
ncclRecv(void* rbuff, size_t count, ncclDataType_t type, int peer, ncclComm_t comm, cudaStream_t stream);

// Collective communication
ncclAllReduce(void* sbuff, void* rbuff, size_t count, ncclDataType_t type, ncclRedOp_t op, ncclComm_t comm, cudaStream_t stream);
ncclBroadcast(void* sbuff, void* rbuff, size_t count, ncclDataType_t type,       int root, ncclComm_t comm, cudaStream_t stream);

// Aggregation/Composition
ncclGroupStart();
ncclGroupEnd();
```

Observe that the communication calls are quite similar in synatx to MPI calls and that they allow using a stream parameter, making them stream-aware.

## Implementation Exercise

Open the [jacobi_nccl.cpp](../../source_code/nccl/jacobi_nccl.cpp) file and understand the flow of the program. Alternatively, you can navigate to `CFD/English/C/source_code/nccl/` directory in Jupyter's file browser in the left pane. Then, click to open the `jacobi_nccl.cpp` file.

Notice how closely it resembles the [jacobi_cuda_aware_mpi.cpp](../../source_code/mpi/jacobi_cuda_aware_mpi.cpp) program that we have used in previous lab.

Also open the [Makefile](../../source_code/nccl/Makefile) and notice that we include NCCL header files in the `mpicxx` build command and we link NCCL libraries by using the `-lnccl` flag.

After the Jacobi device kernel computation, we will use `ncclAllReduce` function to first reduce the (square of) L2 norm in the GPUs and then transfer it to the CPU in each rank. We will use the default stream in this exercise. Recall that the default (or NULL) stream is denoted by "0" and is synchronizing for the device. However, NCCL communication calls are asynchronous and will not block the host.

Implement the following marked as `TODO`:

* Reduce the device-local L2 norm, `l2_norm_d` to the global L2 norm on each device, `l2_global_norm_d`, using `ncclAllReduce()` function. Use `ncclSum` as the reduction operation. Make sure to encapsulate this function call within NCCL group calls, `ncclGroupStart()` and `ncclGroupEnd()`.
* Transfer the global L2 norm from each device to the host using `cudaMemcpyAsync` function.
* Perform the first set of halo exchanges by:
  - Receiving the top halo from the `top` neighbour into the `a_new` device memory array location.
  - Sending current device's bottom halo to `bottom` neighbour from the `a_new + (iy_end - 1) * nx` device memory array location.
* Similarly, perform the second set of halo exchanges.
* For all NCCL calls, use "0" in the stream parameter function argument.
* Synchronize the device before computing the global L2 norm on host for printing.

---

## Running this lab on a single Perlmutter GPU node

This notebook has been adapted from the original bootcamp material, which assumed a DGX cluster with 8 GPUs per node and a `mpirun`-based launcher. **We are running on one Perlmutter GPU node with 4 NVIDIA A100-SXM4-40GB GPUs**, so a few things differ. Read this section before compiling.

### Hardware we are targeting

A Perlmutter GPU node is:

* **one** AMD EPYC 7763 socket (64 cores / 128 hardware threads) — *not* the dual-socket layout the original material assumes, so process-mapping options like `--map-by ppr:N:socket` do not apply here;
* **4x NVIDIA A100-SXM4-40GB**, compute capability **8.0**, all-to-all connected by NVLink3;
* 4x HPE Slingshot 11 NICs (only relevant once you go multi-node).

Because all 4 GPUs sit behind NVLink on a single node, everything we observe in this lab is **intra-node** NCCL behaviour. That is still the heart of the lesson — NCCL's topology detection and its advantage over CUDA-aware MPI — but the inter-node numbers quoted later in this notebook are reference values from other machines, not something we will reproduce here.

Run the cell below to confirm what you have been allocated.

### Changes already made to the Makefile

The upstream `Makefile` targets a CUDA 11.3-era toolchain. On Perlmutter (CUDA 13.2) the following were needed:

1. **Architecture.** `compute_70` (Volta) was removed in CUDA 13, and we only need Ampere anyway, so `GENCODE_FLAGS` is now just `-gencode arch=compute_80,code=sm_80`.
2. **NVTX.** The legacy `nvToolsExt.h` header and `libnvToolsExt.so` are gone; NVTX v3 is header-only and lives under an `nvtx3/` prefix. So `jacobi_nccl.cpp` now does `#include <nvtx3/nvToolsExt.h>` and `-lnvToolsExt` was dropped from `LD_FLAGS`.
3. **rpath.** `-Wl,-rpath,...` entries for CUDA, NCCL and the NCCL plugin directory were added to `LD_FLAGS` so the binary resolves its libraries when `srun` launches it, without depending on `LD_LIBRARY_PATH` propagating.

### NCCL and MPI on Perlmutter

* We build against NERSC's own NCCL module rather than the copy bundled with the HPC SDK, because the NERSC build ships the **aws-ofi-nccl libfabric plugin** (in `$NCCL_HOME/plugin/lib`) that NCCL needs to use the Slingshot network. That matters for multi-node runs; on a single node NCCL will use NVLink regardless.
* `mpicxx` here resolves to **Cray MPICH**, not OpenMPI. This has two consequences:
  * The launcher must be `srun`, and the PMI plugin must be `--mpi=cray_shasta` (the Perlmutter default). Using `--mpi=pmix` makes each rank initialise as its own singleton `MPI_COMM_WORLD` — you will see `Num GPUs: 1.` printed once per rank instead of a single multi-GPU run. If you see duplicated output, that is what happened.
  * Cray MPICH aborts with *"GPU_SUPPORT_ENABLED is requested, but GTL library is not linked"* unless the GPU Transport Layer is linked in. In **this** lab MPI only broadcasts the `ncclUniqueId` and splits the communicator — both host-side operations — so we simply set `MPICH_GPU_SUPPORT_ENABLED=0`. The CUDA-aware MPI lab genuinely needs GPU-aware MPI and must link `-lmpi_gtl_cuda` instead.

### Launching

Every run below uses `srun` into the allocation that already backs this Jupyter server:

```bash
srun --overlap --mpi=cray_shasta -N 1 -n <ranks> -c 32 --gpus-per-task=1 --gpu-bind=none ./jacobi_nccl ...
```

Three flags deserve explanation:

* `--gpu-bind=none` is **required**. `--gpus-per-task=1` implicitly sets `--gpu-bind=per_task`, which would make each rank see exactly one GPU as device 0. This program calls `cudaSetDevice(local_rank)`, so ranks 1-3 would fail. With `--gpu-bind=none` every rank sees all 4 GPUs and picks its own.
* `-c 32` is 128 hardware threads divided by 4 ranks per node.
* `--overlap` lets these job steps share the resources already held by the Jupyter server's own step; without it Slurm may refuse to create the step.

First, set up the environment and compile:


In [1]:
# --- Environment for the NCCL lab on Perlmutter ---------------------------------
# NERSC's NCCL build (matches CUDA 13.2) and its Slingshot/libfabric plugin.
%env NCCL_HOME=/global/common/software/nersc9/nccl/2.29.2-cu13

# This lab uses MPI only for host-side bootstrap (ncclUniqueId broadcast + comm split),
# so Cray MPICH does not need its GPU Transport Layer. See the notes above.
%env MPICH_GPU_SUPPORT_ENABLED=0

env: NCCL_HOME=/global/common/software/nersc9/nccl/2.29.2-cu13
env: MPICH_GPU_SUPPORT_ENABLED=0


In [2]:
# What have we actually been allocated?
!echo "job:   $SLURM_JOB_ID"
!echo "nodes: $SLURM_JOB_NUM_NODES  ($SLURM_JOB_NODELIST)"
!nvidia-smi --query-gpu=index,name,compute_cap --format=csv
!nvidia-smi topo -m

job:   60035022
nodes: 1  (nid008257)
index, name, compute_cap
0, NVIDIA A100-SXM4-80GB, 8.0
1, NVIDIA A100-SXM4-80GB, 8.0
2, NVIDIA A100-SXM4-80GB, 8.0
3, NVIDIA A100-SXM4-80GB, 8.0
	GPU0	GPU1	GPU2	GPU3	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	NV4	NV4	NV4	48-63,112-127	3		N/A
GPU1	NV4	 X 	NV4	NV4	32-47,96-111	2		N/A
GPU2	NV4	NV4	 X 	NV4	16-31,80-95	1		N/A
GPU3	NV4	NV4	NV4	 X 	0-15,64-79	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks


In [3]:
!cd ../../source_code/nccl/ && make clean && make

rm -rf jacobi_nccl jacobi_kernels.o jacobi_nccl.o *.qdrep *.sqlite *.nsys-rep
nvcc -lineinfo  -gencode arch=compute_80,code=sm_80 -std=c++14 jacobi_kernels.cu -c
mpicxx -DUSE_NVTX -I/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/include -I/global/common/software/nersc9/nccl/2.29.2-cu13/include -fopenmp -std=c++14 jacobi_nccl.cpp jacobi_kernels.o -L/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/lib64 -lcudart -L/global/common/software/nersc9/nccl/2.29.2-cu13/lib -lnccl -Wl,-rpath,/opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/lib64 -Wl,-rpath,/global/common/software/nersc9/nccl/2.29.2-cu13/lib -Wl,-rpath,/global/common/software/nersc9/nccl/2.29.2-cu13/plugin/lib -o jacobi_nccl


In [4]:
# Sanity check: libnccl.so.2 must resolve to the NERSC (nersc9) path, not "not found".
!ldd ../../source_code/nccl/jacobi_nccl | grep -i -E 'nccl|cudart' 

	libnccl.so.2 => /global/common/software/nersc9/nccl/2.29.2-cu13/lib/libnccl.so.2 (0x00007fa2d8df6000)
	libcudart.so.13 => /opt/nvidia/hpc_sdk/Linux_x86_64/26.5/cuda/13.2/lib64/libcudart.so.13 (0x00007fa2d6400000)


Ensure that there are no errors, and that `ldd` resolved `libnccl.so.2`.

## Validating the implementation

First a small run on **2 GPUs** with a $2K\times4K$ grid, just to check correctness. The program runs a single-GPU reference first and compares, so you should see the same residual sequence twice and a sensible speedup figure.

A correct run prints **`Num GPUs: 2.`** exactly once. If you instead see the whole run printed twice with `Num GPUs: 1.`, the MPI bootstrap failed — check that `--mpi=cray_shasta` is on the `srun` line.

In [5]:
# 2 GPUs, 2K x 4K grid
!cd ../../source_code/nccl && srun --overlap --mpi=cray_shasta -N 1 -n 2 -c 32 \
    --gpus-per-task=1 --gpu-bind=none ./jacobi_nccl -nx 2048 -ny 4096

srun: warning: can't honor --ntasks-per-node set to 1 which doesn't match the requested tasks 2 with the maximum number of requested nodes 1. Ignoring --ntasks-per-node.
Single GPU jacobi relaxation: 1000 iterations on 2048 x 4096 mesh
  100, 0.452292
  200, 0.268792
  300, 0.198269
  400, 0.159769
  500, 0.135133
  600, 0.117852
  700, 0.104977
  800, 0.094966
  900, 0.086930
 1000, 0.080320
Jacobi relaxation: 1000 iterations on 2048 x 4096 mesh
  100, 0.452292
  200, 0.268793
  300, 0.198269
  400, 0.159769
  500, 0.135134
  600, 0.117852
  700, 0.104977
  800, 0.094966
  900, 0.086930
 1000, 0.080320
Num GPUs: 2.
2048x4096: 1 GPU:   0.1351 s, 2 GPUs:   0.1941 s, speedup:     0.70, efficiency:    34.81 


In [6]:
# All 4 GPUs on the node, 4K x 8K grid (grid scaled with GPU count so each GPU keeps the same work)
!cd ../../source_code/nccl && srun --overlap --mpi=cray_shasta -N 1 -n 4 -c 32 \
    --gpus-per-task=1 --gpu-bind=none ./jacobi_nccl -nx 4096 -ny 8192

srun: warning: can't honor --ntasks-per-node set to 1 which doesn't match the requested tasks 4 with the maximum number of requested nodes 1. Ignoring --ntasks-per-node.
Single GPU jacobi relaxation: 1000 iterations on 4096 x 8192 mesh
  100, 0.639705
  200, 0.380187
  300, 0.280449
  400, 0.226001
  500, 0.191161
  600, 0.166722
  700, 0.148514
  800, 0.134357
  900, 0.122994
 1000, 0.113646
Jacobi relaxation: 1000 iterations on 4096 x 8192 mesh
  100, 0.639705
  200, 0.380187
  300, 0.280449
  400, 0.226001
  500, 0.191162
  600, 0.166723
  700, 0.148514
  800, 0.134357
  900, 0.122994
 1000, 0.113646
Num GPUs: 4.
4096x8192: 1 GPU:   0.4470 s, 4 GPUs:   0.6428 s, speedup:     0.70, efficiency:    17.39 


### Interpreting what you just saw

At only 1000 iterations the numbers are dominated by NCCL start-up cost: the first few collective calls carry a large one-time overhead for topology detection and channel setup, so the reported efficiency will look poor. That is expected and is exactly why the next run uses 5000 iterations.

What to look for:

* **2 GPUs vs 4 GPUs.** The grid was doubled along with the GPU count, so per-GPU work is constant. If efficiency holds up moving from 2 to 4 GPUs, NVLink is absorbing the extra halo traffic well.
* **Speedup below 1 on the short run** is normal and should disappear in the long run below.

Record both results — you will want them for the comparison in the next section.

Now increase the iteration count to 5000 so the fixed start-up cost is amortised. We also pass `-skip_single_gpu` to skip the serial reference run, which otherwise dominates the wall time now that correctness is established.

In [7]:
# 2 GPUs, 5000 iterations
!cd ../../source_code/nccl && srun --overlap --mpi=cray_shasta -N 1 -n 2 -c 32 \
    --gpus-per-task=1 --gpu-bind=none ./jacobi_nccl -nx 2048 -ny 4096 -niter 5000

srun: warning: can't honor --ntasks-per-node set to 1 which doesn't match the requested tasks 2 with the maximum number of requested nodes 1. Ignoring --ntasks-per-node.
Single GPU jacobi relaxation: 5000 iterations on 2048 x 4096 mesh
  100, 0.452292
  200, 0.268792
  300, 0.198269
  400, 0.159769
  500, 0.135133
  600, 0.117852
  700, 0.104977
  800, 0.094966
  900, 0.086930
 1000, 0.080320
 1100, 0.074774
 1200, 0.070045
 1300, 0.065960
 1400, 0.062390
 1500, 0.059240
 1600, 0.056437
 1700, 0.053925
 1800, 0.051659
 1900, 0.049603
 2000, 0.047728
 2100, 0.046010
 2200, 0.044430
 2300, 0.042971
 2400, 0.041618
 2500, 0.040361
 2600, 0.039188
 2700, 0.038092
 2800, 0.037065
 2900, 0.036100
 3000, 0.035192
 3100, 0.034335
 3200, 0.033525
 3300, 0.032758
 3400, 0.032031
 3500, 0.031340
 3600, 0.030683
 3700, 0.030057
 3800, 0.029460
 3900, 0.028890
 4000, 0.028345
 4100, 0.027823
 4200, 0.027323
 4300, 0.026844
 4400, 0.026383
 4500, 0.025941
 4600, 0.025515
 4700, 0.025106
 4800, 0.024

In [8]:
# 4 GPUs, 5000 iterations
!cd ../../source_code/nccl && srun --overlap --mpi=cray_shasta -N 1 -n 4 -c 32 \
    --gpus-per-task=1 --gpu-bind=none ./jacobi_nccl -nx 4096 -ny 8192 -niter 5000

srun: warning: can't honor --ntasks-per-node set to 1 which doesn't match the requested tasks 4 with the maximum number of requested nodes 1. Ignoring --ntasks-per-node.
Single GPU jacobi relaxation: 5000 iterations on 4096 x 8192 mesh
  100, 0.639705
  200, 0.380187
  300, 0.280449
  400, 0.226001
  500, 0.191161
  600, 0.166722
  700, 0.148514
  800, 0.134357
  900, 0.122994
 1000, 0.113646
 1100, 0.105804
 1200, 0.099117
 1300, 0.093340
 1400, 0.088292
 1500, 0.083838
 1600, 0.079875
 1700, 0.076323
 1800, 0.073119
 1900, 0.070212
 2000, 0.067561
 2100, 0.065133
 2200, 0.062898
 2300, 0.060835
 2400, 0.058923
 2500, 0.057145
 2600, 0.055488
 2700, 0.053938
 2800, 0.052486
 2900, 0.051122
 3000, 0.049838
 3100, 0.048626
 3200, 0.047481
 3300, 0.046397
 3400, 0.045369
 3500, 0.044393
 3600, 0.043464
 3700, 0.042580
 3800, 0.041736
 3900, 0.040930
 4000, 0.040159
 4100, 0.039422
 4200, 0.038715
 4300, 0.038037
 4400, 0.037387
 4500, 0.036761
 4600, 0.036160
 4700, 0.035581
 4800, 0.035

Efficiency should now be substantially higher than in the 1000-iteration runs, since the NCCL start-up cost is spread over 5x more iterations.

Compare these figures against the numbers you recorded in the **CUDA-aware MPI** lab for the same GPU count and grid size on this same node. That is the comparison this lab is built around, and on a single NVLink-connected node it is a fair one: same hardware, same grid, same iteration count, only the communication layer differs. NCCL should come out ahead.

## Profiling

Now let us look at where the time actually goes. We profile the 4-GPU run for 5000 iterations with Nsight Systems.

Note the structural change from the original notebook: there, `nsys` wrapped `mpirun`, which worked because `mpirun` was a local process that forked all ranks. With `srun`, `nsys` has to be placed **inside** the launch so that each rank is profiled in its own process. We therefore get one report per rank, named by `SLURM_PROCID`.

In [ ]:
!cd ../../source_code/nccl && export OUT=$SCRATCH/nccl_prof && mkdir -p $OUT && \
 srun --overlap --mpi=cray_shasta -N 1 -n 4 -c 32 --gpus-per-task=1 --gpu-bind=none \
 nsys profile --trace=cuda,nvtx,mpi --mpi-impl=mpich --stats=true --force-overwrite true \
 -o $OUT/jacobi_nccl_%q{SLURM_PROCID} \
 ./jacobi_nccl -nx 4096 -ny 8192 -niter 5000 -skip_single_gpu

In [ ]:
!ls -lh ../../source_code/nccl/output_profiler/

You now have four `.nsys-rep` files, one per rank, in `source_code/nccl/output_profiler/`. Download `jacobi_nccl_0.nsys-rep` (hold <mark>Shift</mark> and <mark>Right-Click</mark> the file in Jupyter's file browser, then "Save Link As") and open it in the Nsight Systems GUI on your laptop.

Note that `--nic-metrics` and the `ucx` trace option from the original notebook have been dropped: the first reports InfiniBand counters and the second requires UCX, neither of which applies to this single-node Slingshot setup.

### What to look for in the timeline

NCCL is already instrumented with NVTX, so you do not need to add your own annotations. However, because NCCL calls are asynchronous with respect to the host and execute mostly on the GPU, the NVTX **Push-Pop** summary in the cell output above is not very informative — the timeline is where the story is.

Zoom into a few consecutive Jacobi iterations and identify:

1. **`ncclKernel_AllReduce...`** — the L2 norm reduction. This should be the shorter of the two.
2. **`ncclKernel_SendRecv...`** — both sets of halo exchanges, aggregated into one kernel because they were wrapped in `ncclGroupStart()`/`ncclGroupEnd()`. This is the dominant communication cost.

Then measure the **idle gap between two successive Jacobi compute kernels** and subtract the communication kernels and the small `cudaMemcpy`/`cudaMemset` calls. The remainder is overhead that NCCL has not been able to hide.

Do the same arithmetic on the profiles you captured in the **CUDA streams/events** and **CUDA-aware MPI** labs. The expected ordering is that NCCL leaves the least idle time of the three, and on this node — where all four GPUs have direct NVLink paths to each other — the gap should be clear.

![nccl_profiler_output](../../images/nccl_profiler_output.png)

*(The screenshot above is from the original material, captured on different hardware. Use it to recognise what the timeline looks like, not as a comparison for your own numbers.)*

**Note:** scrolling through the timeline you will see variation in total and idle times between iterations, for the reasons discussed in previous labs. Judge by the average.

### Things worth trying

* Re-run with `NCCL_DEBUG=INFO` and read the topology NCCL discovers. On this node you should see it detect NVLink between all 4 GPUs. Compare that against the `nvidia-smi topo -m` output from the top of the notebook.
* Disable peer-to-peer with `NCCL_P2P_DISABLE=1` and watch the halo-exchange kernel get slower — this makes NCCL's NVLink usage visible by taking it away.
* Vary the rank count (1, 2, 4) at fixed grid size to see strong scaling rather than the weak scaling we did above.

**Solution:** The solution for this exercise is present in `source_code/nccl/solution` directory: [jacobi_nccl.cpp](../../source_code/nccl/solution/jacobi_nccl.cpp).

We have now learnt about using NCCL with MPI in our application. While the library performs a lot of optimizations under the hood, the API is quite simple and easy-to-use. In applications that require  lot of collective communications, like Fast Fourier Transforms and Deep Learning models, NCCL provides ample opportunity for improving performance.

Now, let us learn about the other important communication-based library that NVIDIA offers, NVSHMEM. Click below to access the next lab:

# [Next: NVSHMEM Library](../nvshmem/nvshmem.ipynb)

Here's a link to the home notebook through which all other notebooks are accessible:

# [HOME](../../../start_here.ipynb)

---
## Links and Resources

* [Concepts: Accelerating IO in data-centers](https://developer.nvidia.com/blog/accelerating-io-in-the-modern-data-center-network-io/)
* [Programming Concepts: NCCL User Guide](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/overview.html)
* [Programming Concepts: Collective communication in NCCL](https://developer.nvidia.com/blog/fast-multi-gpu-collectives-nccl/)
* [Programming Concepts: Scaling Deep Learning programs with NCCL](https://developer.nvidia.com/blog/massively-scale-deep-learning-training-nccl-2-4/)
* [Documentation: NCCL API](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/api.html)
* [Code: Multi-GPU Programming Models](https://github.com/NVIDIA/multi-gpu-programming-models)
* [Code: GPU Bootcamp](https://github.com/gpuhackathons-org/gpubootcamp/)

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

## Licensing
Copyright © 2022 OpenACC-Standard.org.  This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.
